# build User Data

In [ ]:
import json
# 建構使用者
# 使用者內容包括
class User:
    def __init__(self, username, password, email):
        self.username = username
        self.password = password
        self.email = email
# 輸入：num_users：數量, startNumber：從幾號開始取名，取名的方式
def create_users_json(num_users,startNumber):
    users = []
    startNumber-=1
    for i in range(1, num_users+1):
        user = User(str(i+startNumber), str(i+startNumber), f"redacted-user@example.com")
        users.append(user.__dict__)
    return json.dumps(users)

In [ ]:
# 建構 3 個使用者的 JSON 格式變數
json_var = create_users_json(10,1)
print(json_var)

# insert_many data to mongodb

In [ ]:
import pymongo

def insert_data_to_mongodb(data, uri='localhost:27017', database='sample_users', collection='accounts'):
    client = pymongo.MongoClient(uri)
    db = client[database]
    col = db[collection]
    try:
        col.insert_many(data)
        print("Data insertion successful!")
        return True
    except Exception as e:
        print(f"Data insertion failed: {e}")
        return False
    finally:
        client.close()

In [ ]:
# 因為 create_users_json() 返回的是以下 JSON 字串
data_str = create_users_json(1000,1)
# 將 JSON 字串解析為 Python 對象
data = json.loads(data_str)
# insert_many to mongodb
insert_data_to_mongodb(data)

# mongodb opration

In [ ]:
import pymongo

class MongoDB:
    def __init__(self, uri='localhost:27017', database='sample_users', collection='browsing_histories'):
        self.uri = uri
        self.database = database
        self.collection = collection
        self.client = pymongo.MongoClient(self.uri)
        self.db = self.client[self.database]
        self.col = self.db[self.collection]

    def insert_data(self, data):
        try:
            result = self.col.insert_many(data)
            print("Data insertion successful!")
            return result.inserted_ids
        except Exception as e:
            print(f"Data insertion failed: {e}")
            return None
    def insert_data_one(self, data):
        try:
            result = self.col.insert_one(data)
#             print("Data insertion successful!")
            return result.inserted_id
        except Exception as e:
            print(f"Data insertion failed: {e}")
            return None
    def find_data(self, query={}, projection=None, limit=0):
        cursor = self.col.find(query, projection=projection, limit=limit)
#         return cursor
        return [doc for doc in cursor]
    def close(self):
        self.client.close()
    def get_max_EventID0308(self):
        query = {"EventID0308": {"$exists": True}}
        projection = {"EventID0308": 1, "_id": 0}
        cursor = self.col.find(query, projection).sort("EventID0308", pymongo.DESCENDING).limit(1)
        result = list(cursor)
        if result:
            max_event_id = result[0]["EventID0308"]
            return max_event_id
        else:
            print("No EventID0308 found in collection")
            return Non


In [ ]:
    def take_idAndEventID0308FromOrinArticle():
        mongo = MongoDB('localhost:27017', 'sample_event_DB', collection='sample_event_Coll_copy')
        data = mongo.find_data({},{'_id':1,'EventID0308':1},3)
        # max_EventID0308=mongo.get_max_EventID0308()
        mongo.close()

In [ ]:
mongo = MongoDB()
users_id = mongo.find_data({},{'_id':1})
print(users_id)
mongo.close()

In [ ]:
mongo = MongoDB('localhost:27017', 'sample_event_DB', collection='sample_event_Coll_copy')
data = mongo.find_data({'EventID0308':1},{'_id':1,'EventID0308':1})
# max_EventID0308=mongo.get_max_EventID0308()
mongo.close()
print(data)

# build the User Browsing History

In [ ]:
import random
import pymongo
from datetime import datetime
from bson.objectid import ObjectId

class JSONGenerator_UserBrowsingHistory:
    def __init__(self, uri='localhost:27017', database='topic_collection', collection='original'):
        self.uri = uri
        self.database = database
        self.collection = collection
        self.client = pymongo.MongoClient(self.uri)
        self.db = self.client[self.database]
        self.col = self.db[self.collection]
    
    def generate_json(self):
        data = {}
        # get random document from collection
        cursor = self.col.find({}, {'_id': 1, 'EventID0308': 1})
        doc = random.choice([d for d in cursor])
        data['original_id'] = str(doc['_id'])
#         data['EventID'] = str(doc['EventID0308'])
        # generate random view time
        view_time = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%S.%fZ')
        data['view_time'] = view_time #'2022-04-01T12:00:00.000Z'
        # generate random browsing duration
        data['browsing_duration'] = random.randint(0, 30)
        # generate random topic
        data['topic'] = random.randint(0, 3)
        return data
    
    def find_data(self, query={}, projection=None, limit=0):
        cursor = self.col.find(query, projection=projection, limit=limit)
        return [doc for doc in cursor]
    
    def close(self):
        self.client.close()

In [ ]:
# importing ObjectId from bson library
from bson.objectid import ObjectId
json_generator = JSONGenerator_UserBrowsingHistory()
json_data = json_generator.generate_json()
print(f'make a json_data: {json_data}\n')
print(f'json_data type is: {type(json_data)}\n')
print(f'take the Event_id: {json_data["Event_id"]}\n')
print(f'Event_id type is: {type(json_data["Event_id"])}\n')
print(f'take the EventID: {json_data["EventID"]}\n')
print(f'EventID type is: {type(json_data["EventID"])}\n')
query = {"EventID0308": int(json_data["EventID"])} # must be int() when you find with "EventID" !!!
result = json_generator.find_data(query,{'_id':1,'EventID0308':1})
print('result with EventID0308: \n',result)
query = {"_id":  ObjectId(json_data["Event_id"])}
result = json_generator.find_data(query,{'_id':1,'EventID0308':1})  # must be ObjectId() when you find with "EventID" !!!
print('result with _id: \n',result)
print(f'ObjectId type is: {type(ObjectId(json_data["Event_id"]))}')# It is <class 'bson.objectid.ObjectId'> !!! special format
query = {"_id":  ObjectId(json_data["Event_id"])}
result = json_generator.find_data(query) 
print('all result: \n',result)

In [ ]:
from bson.objectid import ObjectId
import random
# take all user _id from mongofb
mongo = MongoDB(uri='localhost:27017', database='sample_users', collection='accounts')
Allusers_id = mongo.find_data({},{'_id':1})
mongo.close()
# destination of storing
mongo = MongoDB(uri='localhost:27017', database='sample_users', collection='browsing_histories')

# print(type(Allusers_id))#<class 'dict'>
# print(len(Allusers_id))# 1000

# counterFor=0
for user_id in Allusers_id:
#     print(type(users_id))#<class 'dict'>
#     if counterFor==3:
#         break
#     counterFor+=1
    # take user _id
    userBrowsingHistory=user_id
    if mongo.find_data({'_id': user_id['_id']}):
        print(f"Data with _id {user_id['_id']} already exists.")
        continue
    # make Browsing History
    json_generator = JSONGenerator_UserBrowsingHistory()
    All_json_data = []
    for xthBrowsingHistory in range(0, random.randint(1, 30)):
        json_data = json_generator.generate_json()
        All_json_data.append(json_data)
    userBrowsingHistory.update({"browsing_history": All_json_data}) 
    mongo.insert_data_one(userBrowsingHistory)
#     print('\n\n',userBrowsingHistory)
mongo.close()

# build matrix(CSC)

In [ ]:
class UserBrowsingHistory:
    def __init__(self, uri='localhost:27017', database='sample_users', collection='browsing_histories'):
        self.uri = uri
        self.database = database
        self.collection = collection
        self.data = None
        
    def load_data(self):
        mongo = MongoDB(uri=self.uri, database=self.database, collection=self.collection)
        data = mongo.find_data(projection={'_id': 1, 'browsing_history.Event_id': 1})
#         print('data:\n',data)
        mongo.close()
        user_textMatrix = {
            'id': [],
            'EventID': [],
            'rowPtr': [0],# 放EventID
            'colInd': [],
            'data': [],
        }
        counterFor = 0
        for doc in data:
            if counterFor == 1:
                break
            counterFor+=1
            user_id = doc['_id']
            browsing_history = doc['browsing_history']            
            for item in browsing_history:
                user_textMatrix[rowInd]
                user_textMatrix['rowInd'].append(1)
        print('rows:\n',rows)
        print('cols:\n',cols)
        print('dataDic:\n',dataDic)

In [ ]:
ubh = UserBrowsingHistory(uri='localhost:27017', database='sample_users', collection='browsing_histories')
mat = ubh.load_data()
print(mat)

In [ ]:
import numpy as np
from scipy.sparse import coo_matrix, csc_matrix

rows = np.array([0, 1, 1, 2, 2, 2])
cols = np.array([0, 1, 2, 0, 1, 2])
data = np.array([1, 2, 3, 4, 5, 6])
coo_mat = coo_matrix((data, (rows, cols)), shape=(3, 3))
csc_mat = csc_matrix(coo_mat)

print(csc_mat)

In [ ]:
user_textMatrix = {
    'rowInd': [],
    'colInd': [],
    'data': [],
}
user_textMatrix['rowInd'].append(1)
print(user_textMatrix)